In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

1. Model Parallelism

In [ ]:
class ModelParallelLLM(nn.Module):
    def __init__(self, vocab_size, num_layers=8, num_gpus=4):
        super().__init__()
        self.num_gpus = num_gpus
        
        layers_per_gpu = num_layers // num_gpus
        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=512,
                    nhead=8,
                    dim_feedforward=2048,
                    dropout=0.1,
                    batch_first=True
                ) for _ in range(layers_per_gpu)
            ]).to(f'cuda:{i}') for i in range(num_gpus)
        ])
    
    def forward(self, x):
        for gpu_layers in self.layers:
            for layer in gpu_layers:
                x = layer(x.to(layer.weight.device))
        return x